## Spark UI — reading the tabs

Once you've localised the slow *task*, the Spark UI is your microscope. Three tabs matter.

- **Jobs tab** — one row per action; the slow job is the slow row.
- **Stages tab** — one row per stage (a stage is the work *between* shuffles); the slow stage is the slow row. Click into it.
- **Stage detail** — the gold. Read two things: the **Tasks table** (a row per task, sortable by duration / shuffle read / spill) and the **task summary metrics** (min, 25th, median, 75th, max for duration, GC, shuffle read/write, spill).

**What each summary signal means:**

| Signal | Diagnosis |
|---|---|
| Max task duration ≫ median | **Skew** — one partition far bigger than the rest |
| Total shuffle write very large | **Shuffle-heavy** — big joins / aggregations |
| Spill (Memory/Disk) > 0 | **Memory pressure** — partitions don't fit in RAM |
| GC time a large fraction of task time | **Memory pressure**, OOM may be imminent |
| Many tiny tasks < 100 ms | **Over-fragmentation** — too many shuffle partitions |

The **Executors tab** shows per-executor memory / disk pressure and any failed / dead executors — a dead executor usually points at OOM.

**Read max-vs-median first.** That one comparison separates skew (max ≫ median) from a uniformly heavy stage (all tasks large) — and those two need completely different fixes.
